# Ejemplo de arquitectura Lakehouse real de empresa para ciencia de datos

## Caso de negocio
Empresa ficticia: `RetailNova`, una cadena de e-commerce y tiendas fisicas que necesita unificar datos operativos y analiticos en una arquitectura Lakehouse.

## Objetivo
Construir un flujo local con capas `Bronze`, `Silver` y `Gold` para:
- Ingerir datos crudos de ventas y clientes.
- Limpiar y estandarizar informacion.
- Generar tablas analiticas para negocio.
- Preparar una tabla de features lista para modelos de ciencia de datos.

## Capas del Lakehouse
- `Bronze`: datos crudos tal como llegan desde sistemas fuente.
- `Silver`: datos limpios, tipados y enriquecidos.
- `Gold`: tablas de negocio y features para analitica y machine learning.

## Resultado esperado
Al final del notebook quedaran creados datasets locales y tablas analiticas reutilizables para dashboards, analisis y modelos predictivos.

## 1. Librerias y estructura del proyecto

En este ejemplo usaremos `pandas` para simular una arquitectura Lakehouse local y reproducible.

In [ ]:
from pathlib import Path
import random
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import pandas as pd

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
BASE_DIR = Path.cwd() / "lakehouse_demo" / RUN_ID
BRONZE_DIR = BASE_DIR / "bronze"
SILVER_DIR = BASE_DIR / "silver"
GOLD_DIR = BASE_DIR / "gold"

for folder in [BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

random.seed(42)
print("Lakehouse base:", BASE_DIR)

## 2. Generar datos empresariales de origen

Simularemos dos fuentes operativas:
- `clientes_raw.csv`: maestro de clientes.
- `ventas_raw.csv`: transacciones de ventas con ruido tipico del mundo real.

In [ ]:
# Generación de datos de clientes y ventas
raw_customers_path = BRONZE_DIR / "clientes_raw.csv"
raw_sales_path = BRONZE_DIR / "ventas_raw.csv"

regions = ["Norte", "Sur", "Centro", "Occidente"]
channels = ["web", "app", "tienda"]
categories = ["electronica", "hogar", "moda", "supermercado"]

clientes = []
for customer_id in range(1, 21):
    clientes.append({
        "customer_id": customer_id,
        "nombre": f"Cliente_{customer_id:03d}",
        "region": random.choice(regions),
        "segmento": random.choice(["pyme", "corporativo", "retail"]),
        "fecha_registro": (datetime(2023, 1, 1) + timedelta(days=random.randint(0, 600))).strftime("%Y-%m-%d")
    })

ventas = []
start_date = datetime(2024, 1, 1)
for sale_id in range(1, 101):
    quantity = random.randint(1, 6)
    unit_price = round(random.uniform(15, 500), 2)
    discount = round(random.choice([0, 0.05, 0.1, 0.15]), 2)
    customer_id = random.randint(1, 20)
    ventas.append({
        "sale_id": sale_id,
        "customer_id": customer_id,
        "fecha_venta": (start_date + timedelta(days=random.randint(0, 180))).strftime("%Y-%m-%d"),
        "canal": random.choice(channels),
        "categoria": random.choice(categories),
        "cantidad": quantity,
        "precio_unitario": unit_price,
        "descuento": discount,
        "estado_pago": random.choice(["pagado", "pagado", "pendiente"]),
        "region": random.choice(regions) if sale_id % 15 != 0 else None
    })

df_clientes_raw = pd.DataFrame(clientes)
df_ventas_raw = pd.DataFrame(ventas)

df_clientes_raw.to_csv(raw_customers_path, index=False)
df_ventas_raw.to_csv(raw_sales_path, index=False)

print(raw_customers_path)
print(raw_sales_path)
df_ventas_raw.head()

## 3. Bronze Layer

> En Bronze almacenamos los datos tal como llegan del sistema fuente, agregando metadatos de ingestión sin alterar la semantica original.

> La ingesta de datos es el proceso de recopilación e importación de archivos de datos 
> de 
> diversas fuentes a una base de datos para su almacenamiento, procesamiento y análisis. > El objetivo de la ingesta de datos es limpiar y almacenar los datos en un repositorio 
> central accesible y coherente para prepararlos para su uso dentro de la organización.

In [ ]:
ingestion_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

bronze_clientes = pd.read_csv(raw_customers_path)
bronze_ventas = pd.read_csv(raw_sales_path)

bronze_clientes["ingestion_ts"] = ingestion_ts
bronze_ventas["ingestion_ts"] = ingestion_ts
bronze_ventas["source_system"] = "erp_ventas"

bronze_clientes_path = BRONZE_DIR / "clientes_bronze.csv"
bronze_ventas_path = BRONZE_DIR / "ventas_bronze.csv"

bronze_clientes.to_csv(bronze_clientes_path, index=False)
bronze_ventas.to_csv(bronze_ventas_path, index=False)

print("Archivos Bronze generados:")
print(bronze_clientes_path)
print(bronze_ventas_path)
bronze_ventas.head()

## 4. Diagnostico de calidad de datos

Antes de pasar a Silver inspeccionamos nulos, duplicados y tipos para detectar problemas frecuentes de integración.

In [ ]:
quality_report = pd.DataFrame({
    "column": bronze_ventas.columns,
    "nulls": bronze_ventas.isna().sum().values,
    "dtype": bronze_ventas.dtypes.astype(str).values
})

quality_report

## 5. Silver Layer

En Silver limpiamos, tipamos y enriquecemos la información para volverla confiable y reusable por analistas, científicos de datos y equipos de negocio.

In [ ]:
silver_clientes = bronze_clientes.copy()
silver_ventas = bronze_ventas.copy()

silver_clientes["fecha_registro"] = pd.to_datetime(silver_clientes["fecha_registro"], errors="coerce")
silver_ventas["fecha_venta"] = pd.to_datetime(silver_ventas["fecha_venta"], errors="coerce")

silver_ventas["region"] = silver_ventas["region"].fillna("Sin region")
silver_ventas = silver_ventas.drop_duplicates(subset=["sale_id"])
silver_ventas = silver_ventas[silver_ventas["estado_pago"].isin(["pagado", "pendiente"])]

silver_ventas["cantidad"] = pd.to_numeric(silver_ventas["cantidad"], errors="coerce").fillna(0)
silver_ventas["precio_unitario"] = pd.to_numeric(silver_ventas["precio_unitario"], errors="coerce").fillna(0)
silver_ventas["descuento"] = pd.to_numeric(silver_ventas["descuento"], errors="coerce").fillna(0)

silver_ventas["importe_bruto"] = silver_ventas["cantidad"] * silver_ventas["precio_unitario"]
silver_ventas["importe_neto"] = silver_ventas["importe_bruto"] * (1 - silver_ventas["descuento"])

silver_ventas_full = silver_ventas.merge(
    silver_clientes[["customer_id", "nombre", "segmento", "region"]].rename(columns={"region": "region_cliente"}),
    on="customer_id",
    how="left"
    )

silver_clientes_path = SILVER_DIR / "clientes_silver.csv"
silver_ventas_path = SILVER_DIR / "ventas_silver.csv"
silver_ventas_full_path = SILVER_DIR / "ventas_clientes_silver.csv"

silver_clientes.to_csv(silver_clientes_path, index=False)
silver_ventas.to_csv(silver_ventas_path, index=False)
silver_ventas_full.to_csv(silver_ventas_full_path, index=False)

silver_ventas_full.head()

## 6. Gold Layer

En Gold construimos tablas orientadas a decisiones: KPIs por región, desempeño comercial y una tabla de features para modelos.

In [ ]:
gold_sales_region = (
    silver_ventas_full
    .groupby("region", as_index=False)
    .agg(
        ventas_total=("sale_id", "count"),
        ingreso_neto=("importe_neto", "sum"),
        ticket_promedio=("importe_neto", "mean")
    )
    .sort_values("ingreso_neto", ascending=False)
    )

gold_sales_category = (
    silver_ventas_full
    .groupby(["categoria", "canal"], as_index=False)
    .agg(
        ventas_total=("sale_id", "count"),
        ingreso_neto=("importe_neto", "sum")
    )
    .sort_values("ingreso_neto", ascending=False)
    )

gold_sales_region.to_csv(GOLD_DIR / "gold_sales_region.csv", index=False)
gold_sales_category.to_csv(GOLD_DIR / "gold_sales_category.csv", index=False)

gold_sales_region

## 7. Feature Store para ciencia de datos

Creamos una tabla de features a nivel cliente para casos como `churn`, `propension de compra` o `valor futuro del cliente`.

# propension de compra : probabilidad de que un cliente adquiera un producto o servicio, medida mediante algoritmos de machine learning y datos históricos
# churn : tasa de abandono
# valor futuro del cliente (Customer Lifetime Value, CLV/LTV) :  métrica predictiva que estima el beneficio neto total que un cliente aportará a una empresa durante toda su relación comercial

In [ ]:
# Análisis de clientes de alto valor

fecha_corte = silver_ventas_full["fecha_venta"].max()

customer_features = (
    silver_ventas_full
    .groupby("customer_id", as_index=False)
    .agg(
        total_compras=("sale_id", "count"),
        gasto_total=("importe_neto", "sum"),
        ticket_promedio=("importe_neto", "mean"),
        descuento_promedio=("descuento", "mean"),
        fecha_ultima_compra=("fecha_venta", "max")
    )
    )

customer_features["dias_desde_ultima_compra"] = (
    fecha_corte - customer_features["fecha_ultima_compra"]
    ).dt.days

customer_features = customer_features.merge(
    silver_clientes[["customer_id", "segmento", "region"]],
    on="customer_id",
    how="left"
    )

customer_features["target_alto_valor"] = (customer_features["gasto_total"] >= customer_features["gasto_total"].median()).astype(int)

customer_features.to_csv(GOLD_DIR / "customer_features.csv", index=False)
customer_features.head()

In [ ]:
# Tarjeta de KPIs ejecutivos
kpis = {
    "ingreso_total": round(gold_sales_region["ingreso_neto"].sum(), 2),
    "ticket_promedio_global": round(silver_ventas_full["importe_neto"].mean(), 2),
    "clientes_alto_valor": int(customer_features["target_alto_valor"].sum()),
    "porcentaje_alto_valor": round(customer_features["target_alto_valor"].mean() * 100, 2)
}

kpi_df = pd.DataFrame([kpis])
kpi_df

In [ ]:
# Validar dependencias de las celdas anteriores
required_tables = ["gold_sales_region", "gold_sales_category", "customer_features"]
missing = [name for name in required_tables if name not in globals()]
if missing:
    raise ValueError(f"Faltan tablas en memoria: {missing}. Ejecuta las celdas de Gold y Feature Store.")

# Estilo ejecutivo
plt.style.use("ggplot")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Lakehouse Ejecutivo - RetailNova", fontsize=16, fontweight="bold")

# 1) Ingreso neto por region
region_df = gold_sales_region.sort_values("ingreso_neto", ascending=False)
axes[0, 0].bar(region_df["region"], region_df["ingreso_neto"], color="#1f77b4")
axes[0, 0].set_title("Ingreso neto por region")
axes[0, 0].set_ylabel("Ingreso neto")
axes[0, 0].tick_params(axis="x", rotation=30)

# 2) Ventas por canal
canal_df = gold_sales_category.groupby("canal", as_index=False)["ventas_total"].sum()
axes[0, 1].bar(canal_df["canal"], canal_df["ventas_total"], color="#2ca02c")
axes[0, 1].set_title("Volumen de ventas por canal")
axes[0, 1].set_ylabel("Numero de ventas")

# 3) Top 10 clientes por gasto total
top_customers = customer_features.sort_values("gasto_total", ascending=False).head(10)
axes[1, 0].bar(top_customers["customer_id"].astype(str), top_customers["gasto_total"], color="#ff7f0e")
axes[1, 0].set_title("Top 10 clientes por gasto total")
axes[1, 0].set_ylabel("Gasto total")
axes[1, 0].tick_params(axis="x", rotation=45)

# 4) Distribucion de recencia
axes[1, 1].hist(customer_features["dias_desde_ultima_compra"], bins=10, color="#9467bd", edgecolor="black")
axes[1, 1].set_title("Distribucion de recencia (dias)")
axes[1, 1].set_xlabel("Dias desde ultima compra")
axes[1, 1].set_ylabel("Cantidad de clientes")

plt.tight_layout()
plt.show()

## 8. Graficos ejecutivos

Estos graficos resumen el rendimiento comercial y la base de clientes para toma de decisiones ejecutivas.

## 9. Conclusiones

Con esta estructura:
- Operaciones entrega datos crudos en Bronze.
- Analitica consume tablas confiables desde Silver.
- Negocio y ciencia de datos trabajan sobre Gold y `customer_features`.
- Liderazgo puede monitorear KPIs y tendencias en los graficos ejecutivos.

Este es el patron base de una arquitectura Lakehouse moderna, aunque en un entorno empresarial normalmente se implementa con formatos como Delta/Parquet, orquestacion y catalogo de datos.


Los formatos de almacenamiento de datos modernos, similares a Delta Lake y Parquet, se enfocan en la eficiencia columnar, la compresión y la compatibilidad con el paradigma lakehouse. Las principales alternativas incluyen Apache Iceberg, Apache Hudi y ORC. Estos formatos permiten transacciones ACID, evolución de esquemas y alta velocidad en analítica de datos

In [ ]:
print("Archivos generados en el Lakehouse demo:")
for path in sorted(BASE_DIR.rglob("*.csv")):
    print(path.relative_to(BASE_DIR))